# Store Sales Database and Report

*Combine related tables, targeted changes, SQL summaries, pandas reports, and NumPy comparisons in a saved workflow.*


## Store Records

The store database starts with five sales. A separate lookup table
records Mina as the owner of Accessory, Kevin as the owner of Device,
and Jisoo as the owner of Display.

| Sale ID | Product | Category | Quantity | Unit price (dollars) | Month |
|---:|---|---|---:|---:|---:|
| 1 | Mouse | Accessory | 3 | 25 | 1 |
| 2 | Keyboard | Accessory | 2 | 45 | 1 |
| 3 | Monitor | Display | 1 | 180 | 2 |
| 4 | Laptop Stand | Accessory | 4 | 30 | 2 |
| 5 | Tablet | Device | 2 | 220 | 3 |

Three changes produce the final records:

- Webcam is added to Accessory with quantity 3, unit price 70, and month 3.
- Keyboard, sale ID 2, receives a corrected unit price of 50.
- Monitor, sale ID 3, is removed.

The report combines each category with its owner, summarizes category
and monthly revenue, and identifies categories reaching 500 dollars.
The database and CSV report are saved under `outputs/08_store_sales/`
and reopened to verify their contents.


## Imports, Prepared Rows, and Output Paths

The notebook contains all input data. The fixed output paths make the
saved files easy to inspect. Relative paths use this chapter folder
locally and `/home/pyodide` in the course site's browser filesystem.
`contextlib.closing()` will ensure database connections close when their
blocks finish; explicit `commit()` still controls saved changes.


In [1]:
import os
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd

output_directory = os.path.join("outputs", "08_store_sales")
database_path = os.path.join(output_directory, "store_sales.db")
report_path = os.path.join(output_directory, "category_revenue.csv")
os.makedirs(output_directory, exist_ok=True)

category_rows = [("Accessory", "Mina"), ("Device", "Kevin"), ("Display", "Jisoo")]
sales_rows = [
    ("Mouse", "Accessory", 3, 25, 1),
    ("Keyboard", "Accessory", 2, 45, 1),
    ("Monitor", "Display", 1, 180, 2),
    ("Laptop Stand", "Accessory", 4, 30, 2),
    ("Tablet", "Device", 2, 220, 3),
]


## Initialize the Related Tables

Reset the child `sales` table before its parent `category_owners` table.
Recreate the foreign key and insert parent records before child records.
This reset applies only to the two tables in this program-owned database.


In [2]:
def initialize_database(connection):
    connection.execute("PRAGMA foreign_keys = ON")
    connection.execute("DROP TABLE IF EXISTS sales")
    connection.execute("DROP TABLE IF EXISTS category_owners")
    connection.execute("""
    CREATE TABLE category_owners (
        category TEXT PRIMARY KEY NOT NULL,
        owner TEXT NOT NULL
    )
    """)
    connection.execute("""
    CREATE TABLE sales (
        sale_id INTEGER PRIMARY KEY,
        product_name TEXT NOT NULL,
        category TEXT NOT NULL REFERENCES category_owners(category),
        quantity INTEGER NOT NULL,
        unit_price INTEGER NOT NULL,
        month INTEGER NOT NULL
    )
    """)
    connection.executemany(
        "INSERT INTO category_owners (category, owner) VALUES (?, ?)", category_rows
    )
    connection.executemany("""
    INSERT INTO sales (product_name, category, quantity, unit_price, month)
    VALUES (?, ?, ?, ?, ?)
    """, sales_rows)
    connection.commit()


## Apply the Recorded Changes

Use primary keys to select Keyboard and Monitor. These three statements
form one transaction. Commit after all three have run so the report reads
the final stored state.


In [3]:
def apply_changes(connection):
    connection.execute("""
    INSERT INTO sales (product_name, category, quantity, unit_price, month)
    VALUES (?, ?, ?, ?, ?)
    """, ("Webcam", "Accessory", 3, 70, 3))
    connection.execute(
        "UPDATE sales SET unit_price = ? WHERE sale_id = ?", (50, 2)
    )
    connection.execute("DELETE FROM sales WHERE sale_id = ?", (3,))
    connection.commit()


## Define the Reports in SQL

The detail query preserves each sale's ID and calculates its revenue.
The category query joins the owner lookup and groups by the selected
category and owner. `HAVING` accepts a bound revenue threshold: zero
includes all categories represented by these retained sales, and 500
selects qualifying groups. The inner join does not create a zero-sales
row for Display after Monitor is removed.

The monthly query explicitly orders the month numbers so adjacent NumPy
values will represent successive months in this prepared case.


In [4]:
detail_query = """
SELECT sale_id, product_name, quantity, unit_price, month,
       quantity * unit_price AS revenue
FROM sales
ORDER BY sale_id
"""

category_query = """
SELECT s.category, c.owner,
       COUNT(*) AS sale_count,
       SUM(s.quantity * s.unit_price) AS total_revenue
FROM sales AS s
JOIN category_owners AS c ON s.category = c.category
GROUP BY s.category, c.owner
HAVING SUM(s.quantity * s.unit_price) >= ?
ORDER BY total_revenue DESC, s.category ASC
"""

monthly_query = """
SELECT month, SUM(quantity * unit_price) AS total_revenue
FROM sales
GROUP BY month
ORDER BY month
"""


## Read and Display the Reports

`read_reports()` receives the final detail rows and grouped views as
DataFrames. `display_reports()` prints those results, compares revenue
totals, and uses `np.diff()` to calculate adjacent monthly changes.
The sum of all detail revenue should agree with both grouped totals
because each retained sale contributes once to each report.


In [5]:
def read_reports(connection):
    details = pd.read_sql_query(detail_query, connection)
    categories = pd.read_sql_query(category_query, connection, params=(0,))
    qualifying = pd.read_sql_query(category_query, connection, params=(500,))
    monthly = pd.read_sql_query(monthly_query, connection)
    return details, categories, qualifying, monthly


def display_reports(details, categories, qualifying, monthly):
    print("Final sales:")
    print(details)
    print("Category report (dollars):")
    print(categories)
    print("Categories reaching 500 dollars:")
    print(qualifying)
    print("Monthly report (dollars):")
    print(monthly)

    monthly_values = monthly["total_revenue"].to_numpy()
    grand_total = details["revenue"].sum()
    totals_agree = (
        grand_total == categories["total_revenue"].sum() == monthly_values.sum()
    )
    print("Monthly changes (dollars):", np.diff(monthly_values))
    print("Final row count:", len(details))
    print("Grand total:", grand_total)
    print("Totals agree:", totals_agree)


## Save the CSV and Reopen Both Files

Save the category report without a pandas index column. Read that CSV
back and open a new connection to the saved database. Compare both the
individual sales and the category totals with the in-memory reports;
these comparisons verify that the committed records persist.


In [6]:
def save_and_verify(details, categories):
    categories.to_csv(report_path, index=False, encoding="utf-8")
    reopened_report = pd.read_csv(report_path, encoding="utf-8")

    with closing(sqlite3.connect(database_path)) as reopened_connection:
        reopened_details = pd.read_sql_query(detail_query, reopened_connection)
        reopened_categories = pd.read_sql_query(
            category_query, reopened_connection, params=(0,)
        )

    print("Saved database:", database_path)
    print("Saved report:", report_path)
    print("CSV report matches:", reopened_report.equals(categories))
    print("Reopened database sales match:", reopened_details.equals(details))
    print("Reopened database report matches:", reopened_categories.equals(categories))
    print("Reopened database row count:", len(reopened_details))


## Run the Complete Case

`main()` creates the prepared tables, applies the three recorded changes,
and reads the reports. The connection closes before the reporting and
file verification steps. Calling the function again resets the generated
tables and replaces the same CSV file.


In [7]:
def main():
    with closing(sqlite3.connect(database_path)) as connection:
        initialize_database(connection)
        apply_changes(connection)
        details, categories, qualifying, monthly = read_reports(connection)

    display_reports(details, categories, qualifying, monthly)
    save_and_verify(details, categories)
    return details, categories, monthly


final_sales_df, category_report_df, monthly_report_df = main()


Final sales:
   sale_id  product_name  quantity  unit_price  month  revenue
0        1         Mouse         3          25      1       75
1        2      Keyboard         2          50      1      100
2        4  Laptop Stand         4          30      2      120
3        5        Tablet         2         220      3      440
4        6        Webcam         3          70      3      210
Category report (dollars):
    category  owner  sale_count  total_revenue
0  Accessory   Mina           4            505
1     Device  Kevin           1            440
Categories reaching 500 dollars:
    category owner  sale_count  total_revenue
0  Accessory  Mina           4            505
Monthly report (dollars):
   month  total_revenue
0      1            175
1      2            120
2      3            650
Monthly changes (dollars): [-55 530]
Final row count: 5
Grand total: 945
Totals agree: True
Saved database: outputs/08_store_sales/store_sales.db
Saved report: outputs/08_store_sales/category_re

## Result Analysis

The final database contains **5 sales** with IDs 1, 2, 4, 5, and 6. Relative to the original 905 dollars, Webcam
adds 210, Keyboard's correction adds 10, and deleting Monitor removes 180.
Final revenue is therefore **945 dollars**.

Accessory, owned by Mina, has four records and revenue **505 dollars**.
Device, owned by Kevin, has one record and revenue **440 dollars**.
Only Accessory reaches 500. The Display lookup remains in its own table,
but no retained sale contributes a Display row to this inner-join report.

Monthly totals are **175, 120, and 650 dollars** for months 1, 2, and 3.
Their adjacent differences are **-55 and 530 dollars**. Detail, category,
and monthly totals agree. The CSV report, reopened database sales, and
reopened database report comparisons are all `True`. The reopened
database contains five sales.

Repeating the full workflow resets the example tables and replaces the
CSV report, so the same prepared case yields the same results.
